In [1]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')


!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

import zipfile
with zipfile.ZipFile("challenges-in-representation-learning-facial-expression-recognition-challenge.zip", "r") as zip_ref:
    zip_ref.extractall("fer2013_data")

print("Dataset downloaded and extracted successfully!")


100% 285M/285M [00:01<00:00, 171MB/s]

Dataset downloaded and extracted successfully!


In [2]:
import os

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class FERDataset(Dataset):
  def __init__(self, X, y):
      self.X = X.reset_index(drop=True)
      self.y = y.reset_index(drop=True)

  def __len__(self):
      return len(self.X)

  def __getitem__(self, idx):
      pixels = np.fromstring(self.X.iloc[idx], sep=' ', dtype=np.uint8)
      img = pixels.reshape(48, 48)
      img = torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0
      label = torch.tensor(self.y.iloc[idx], dtype=torch.long)
      return img, label


In [4]:


import pandas as pd

train_split = pd.read_csv('/content/drive/MyDrive/FER_project/data/train_split.csv')
val_split = pd.read_csv('/content/drive/MyDrive/FER_project/data/val_split.csv')





In [5]:

print(train_split.shape)
print(val_split.shape)

(22967, 2)
(5742, 2)


In [6]:


X_train = train_split["pixels"]
y_train = train_split["emotion"]

X_val = val_split["pixels"]
y_val = val_split["emotion"]





In [7]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

(22967,)
(22967,)
(5742,)
(5742,)


In [8]:

train_dataset = FERDataset(X_train, y_train)
val_dataset = FERDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [9]:
import torch
import torch.nn as nn


class MediumCNN(nn.Module):
    def __init__(self):
        super(MediumCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),


            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [10]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MediumCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [11]:
!pip install wandb -q

In [12]:

import wandb

wandb.login()



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nmetr23 (nmetr23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)



In [15]:
def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    val_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds==y).sum().item()
            total += y.size(0)

    return val_loss / len(loader), correct / total





In [16]:
lrs = [1e-3, 3e-4, 1e-4]
batch_sizes = [10, 32, 64]
optimizers = ["adam", "sgd"]


In [17]:
import itertools
import torch


for lr, batch_size, opt_name in itertools.product(lrs, batch_sizes, optimizers):

    run_name = f"MediumCNN (medium_lr{lr}_bs{batch_size}_opt{opt_name})"

    wandb.init(
        project="FER-Challenge",
        name=run_name,
        reinit=True
    )

    wandb.config.update({
        "lr": lr,
        "batch_size": batch_size,
        "optimizer": opt_name,
        "model": "MediumCNN"
    })

    model = MediumCNN().to(device)

    if opt_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    criterion = torch.nn.CrossEntropyLoss()

    epochs = 5

    for epoch in range(epochs):

        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        })

        print(f"{run_name} | Epoch {epoch+1}: loss={train_loss:.4f}, acc={val_acc:.4f}")

    wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


MediumCNN (medium_lr0.001_bs10_optadam) | Epoch 1: loss=1.7604, acc=0.3844
MediumCNN (medium_lr0.001_bs10_optadam) | Epoch 2: loss=1.5518, acc=0.3805
MediumCNN (medium_lr0.001_bs10_optadam) | Epoch 3: loss=1.4665, acc=0.4770
MediumCNN (medium_lr0.001_bs10_optadam) | Epoch 4: loss=1.4029, acc=0.4626
MediumCNN (medium_lr0.001_bs10_optadam) | Epoch 5: loss=1.3559, acc=0.4540


epoch,▁▃▅▆█
train_loss,█▄▃▂▁
val_accuracy,▁▁█▇▆
val_loss,██▃▁▃
epoch,5
train_loss,1.35586
val_accuracy,0.45402
val_loss,1.41572


MediumCNN (medium_lr0.001_bs10_optsgd) | Epoch 1: loss=1.7915, acc=0.2980
MediumCNN (medium_lr0.001_bs10_optsgd) | Epoch 2: loss=1.7307, acc=0.3231
MediumCNN (medium_lr0.001_bs10_optsgd) | Epoch 3: loss=1.6903, acc=0.3657
MediumCNN (medium_lr0.001_bs10_optsgd) | Epoch 4: loss=1.6544, acc=0.3863
MediumCNN (medium_lr0.001_bs10_optsgd) | Epoch 5: loss=1.6182, acc=0.3830


epoch,▁▃▅▆█
train_loss,█▆▄▂▁
val_accuracy,▁▃▆██
val_loss,█▆▄▂▁
epoch,5
train_loss,1.61817
val_accuracy,0.38297
val_loss,1.58399


...

epoch,▁▃▅▆█
train_loss,█▄▃▂▁
val_accuracy,▁▂▄▅█
val_loss,█▅▃▂▁
epoch,5
train_loss,1.7929
val_accuracy,0.27395
val_loss,1.76234


MediumCNN (medium_lr0.0001_bs64_optadam) | Epoch 1: loss=1.6823, acc=0.4284
MediumCNN (medium_lr0.0001_bs64_optadam) | Epoch 2: loss=1.5021, acc=0.4732
MediumCNN (medium_lr0.0001_bs64_optadam) | Epoch 3: loss=1.4099, acc=0.4711
MediumCNN (medium_lr0.0001_bs64_optadam) | Epoch 4: loss=1.3450, acc=0.5017
MediumCNN (medium_lr0.0001_bs64_optadam) | Epoch 5: loss=1.2900, acc=0.5152


epoch,▁▃▅▆█
train_loss,█▅▃▂▁
val_accuracy,▁▅▄▇█
val_loss,█▅▃▂▁
epoch,5
train_loss,1.29001
val_accuracy,0.51515
val_loss,1.27535


MediumCNN (medium_lr0.0001_bs64_optsgd) | Epoch 1: loss=1.9000, acc=0.2489
MediumCNN (medium_lr0.0001_bs64_optsgd) | Epoch 2: loss=1.8401, acc=0.2565
MediumCNN (medium_lr0.0001_bs64_optsgd) | Epoch 3: loss=1.8177, acc=0.2602
MediumCNN (medium_lr0.0001_bs64_optsgd) | Epoch 4: loss=1.7994, acc=0.2692
MediumCNN (medium_lr0.0001_bs64_optsgd) | Epoch 5: loss=1.7925, acc=0.2797


epoch,▁▃▅▆█
train_loss,█▄▃▁▁
val_accuracy,▁▃▄▆█
val_loss,█▅▃▂▁
epoch,5
train_loss,1.79254
val_accuracy,0.27969
val_loss,1.76315
